<a href="https://colab.research.google.com/github/maplesugano/NLP_IndividualCW/blob/main/Reconstruct_and_RoBERTa_baseline_train_dev_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Main imports and code

In [1]:
# check which gpu we're using
!nvidia-smi

Tue Mar  3 12:57:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.65.06              Driver Version: 580.65.06      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080        Off |   00000000:2D:00.0  On |                  N/A |
|  0%   39C    P8              3W /  320W |   12272MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers==4.39.3 accelerate==0.30.1 scikit-learn tensorflow

In [3]:
from urllib import request
import logging
from collections import Counter
from ast import literal_eval
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
 )
from sklearn.metrics import accuracy_score, f1_score

/vol/bitbucket/ks2222/NLP_IndividualCW/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-03 12:58:04.310224: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256, label_dtype=torch.long):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length,
        )
        self.labels = labels
        self.label_dtype = label_dtype

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=self.label_dtype)
        return item

def compute_metrics_binary(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

def compute_metrics_multilabel(eval_pred):
    logits, labels = eval_pred
    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

/vol/bitbucket/ks2222/NLP_IndividualCW/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
# prepare logger
logging.basicConfig(level=logging.INFO)

transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

# check gpu
cuda_available = torch.cuda.is_available()

print('Cuda available? ',cuda_available)

Cuda available?  True


In [6]:
if cuda_available:
  import tensorflow as tf
  # Get the GPU device name.
  device_name = tf.test.gpu_device_name()
  # The device name should look like the following:
  if device_name == '/device:GPU:0':
      print('Found GPU at: {}'.format(device_name))
  else:
      raise SystemError('GPU device not found')

Found GPU at: /device:GPU:0


I0000 00:00:1772542697.077645 3334379 gpu_device.cc:2020] Created device /device:GPU:0 with 1893 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080, pci bus id: 0000:2d:00.0, compute capability: 8.9


# Fetch Don't Patronize Me! data manager module

In [7]:
module_url = f"https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/dont_patronize_me.py"
module_name = module_url.split('/')[-1]
print(f'Fetching {module_url}')
#with open("file_1.txt") as f1, open("file_2.txt") as f2
with request.urlopen(module_url) as f, open(module_name,'w') as outf:
  a = f.read()
  outf.write(a.decode('utf-8'))

Fetching https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/dont_patronize_me.py


In [8]:
# helper function to save predictions to an output file
def labels2file(p, outf_path):
	with open(outf_path,'w') as outf:
		for pi in p:
			outf.write(','.join([str(k) for k in pi])+'\n')

In [9]:
from dont_patronize_me import DontPatronizeMe

In [10]:
dpm = DontPatronizeMe('.', '.')

In [11]:
pcl_url = "https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv"
pcl_file = "dontpatronizeme_pcl.tsv"
category_url = "https://github.com/CRLala/NLPLabs-2024/blob/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_categories.tsv"
category_file = "dontpatronizeme_categories.tsv"

with request.urlopen(pcl_url) as resp, open(pcl_file, "wb") as out:
    out.write(resp.read())

print(f"Downloaded {pcl_file}")

with request.urlopen(category_url) as resp, open(category_file, "wb") as out:
    out.write(resp.read())

print(f"Downloaded {category_file}")

Downloaded dontpatronizeme_pcl.tsv
Downloaded dontpatronizeme_categories.tsv


In [12]:
pcl_url = "https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv"
pcl_file = "dontpatronizeme_pcl.tsv"
category_url = "https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_categories.tsv"
category_file = "dontpatronizeme_categories.tsv"

with request.urlopen(pcl_url) as resp, open(pcl_file, "wb") as out:
    out.write(resp.read())
print(f"Downloaded {pcl_file}")

with request.urlopen(category_url) as resp, open(category_file, "wb") as out:
    out.write(resp.read())
print(f"Downloaded {category_file}")

Downloaded dontpatronizeme_pcl.tsv
Downloaded dontpatronizeme_categories.tsv


# Load paragraph IDs

In [13]:
dev_url = "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/dev_semeval_parids-labels.csv"
dev_file = "dev_semeval_parids-labels.csv"
train_url = "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/train_semeval_parids-labels.csv"
train_file = "train_semeval_parids-labels.csv"

with request.urlopen(dev_url) as resp, open(dev_file, "wb") as out:
    out.write(resp.read())
print(f"Downloaded {dev_file}")

with request.urlopen(train_url) as resp, open(train_file, "wb") as out:
    out.write(resp.read())
print(f"Downloaded {train_file}")

Downloaded dev_semeval_parids-labels.csv
Downloaded train_semeval_parids-labels.csv


In [14]:
trids = pd.read_csv('train_semeval_parids-labels.csv')
teids = pd.read_csv('dev_semeval_parids-labels.csv')

In [15]:
trids.par_id = trids.par_id.astype(str)
teids.par_id = teids.par_id.astype(str)

In [16]:
if dpm.train_task1_df is None:
    dpm.load_task1()
data = dpm.train_task1_df

In [17]:
data

,par_id,art_id,keyword,country,text,label,orig_label
0,1,@@24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0,0
1,2,@@21968160,migrant,gh,"In Libya today , there are countless number of...",0,0
2,3,@@16584954,immigrant,ie,"""White House press secretary Sean Spicer said ...",0,0
3,4,@@7811231,disabled,nz,Council customers only signs would be displaye...,0,0
4,5,@@1494111,refugee,ca,""""""" Just like we received migrants fleeing El ...",0,0
...,...,...,...,...,...,...,...
10464,10465,@@14297363,women,lk,"""Sri Lankan norms and culture inhibit women fr...",0,1
10465,10466,@@70091353,vulnerable,ph,He added that the AFP will continue to bank on...,0,0
10466,10467,@@20282330,in-need,ng,""""""" She has one huge platform , and informatio...",1,3
10467,10468,@@16753236,hopeless,in,""""""" Anja Ringgren Loven I ca n't find a word t...",1,4




# Rebuild training set (Task 1)

In [18]:
rows = [] # will contain par_id, label and text
for idx in range(len(trids)):
  parid = trids.par_id[idx]
  #print(parid)
  # select row from original dataset to retrieve `text` and binary label
  keyword = data.loc[data.par_id == parid].keyword.values[0]
  text = data.loc[data.par_id == parid].text.values[0]
  label = data.loc[data.par_id == parid].label.values[0]
  rows.append({
      'par_id':parid,
      'community':keyword,
      'text':text,
      'label':label
  })


In [19]:
import random

In [20]:
trdf1 = pd.DataFrame(rows)

In [21]:
trdf1

,par_id,community,text,label
0,4341,poor-families,"The scheme saw an estimated 150,000 children f...",1
1,4136,homeless,Durban 's homeless communities reconciliation ...,1
2,10352,poor-families,The next immediate problem that cropped up was...,1
3,8279,vulnerable,Far more important than the implications for t...,1
4,1164,poor-families,To strengthen child-sensitive social protectio...,1
...,...,...,...,...
8370,8380,refugee,Rescue teams search for survivors on the rubbl...,0
8371,8381,hopeless,The launch of ' Happy Birthday ' took place la...,0
8372,8382,homeless,"The unrest has left at least 20,000 people dea...",0
8373,8383,hopeless,You have to see it from my perspective . I may...,0


# Rebuild test set (Task 1)

In [22]:
rows = [] # will contain par_id, label and text
for idx in range(len(teids)):
  parid = teids.par_id[idx]
  #print(parid)
  # select row from original dataset
  keyword = data.loc[data.par_id == parid].keyword.values[0]
  text = data.loc[data.par_id == parid].text.values[0]
  label = data.loc[data.par_id == parid].label.values[0]
  rows.append({
      'par_id':parid,
      'community':keyword,
      'text':text,
      'label':label
  })


In [23]:
len(rows)

2094

In [24]:
tedf1 = pd.DataFrame(rows)

In [25]:
tedf1 = tedf1.sample(frac=1, random_state=42).reset_index(drop=True)

# RoBERTa Baseline for Task 1

In [26]:
# downsample negative instances
pcldf = trdf1[trdf1.label==1]
npos = len(pcldf)

training_set1 = pd.concat([pcldf,trdf1[trdf1.label==0][:npos*2]])

In [27]:
training_set1

,par_id,community,text,label
0,4341,poor-families,"The scheme saw an estimated 150,000 children f...",1
1,4136,homeless,Durban 's homeless communities reconciliation ...,1
2,10352,poor-families,The next immediate problem that cropped up was...,1
3,8279,vulnerable,Far more important than the implications for t...,1
4,1164,poor-families,To strengthen child-sensitive social protectio...,1
...,...,...,...,...
2377,1775,refugee,Last but not the least element of culpability ...,0
2378,1776,refugee,"Then , taking the art of counter-intuitive non...",0
2379,1777,refugee,Kagunga village was reported to lack necessary...,0
2380,1778,vulnerable,"""After her parents high-profile divorce after ...",0


In [ ]:
training_args_task1 = TrainingArguments(
    output_dir="./tmp_task1",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=50,
    save_strategy="no",
    evaluation_strategy="no",
    report_to="none",
 )
model_task1 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
 )
train_dataset_task1 = TextDataset(
    training_set1["text"].tolist(),
    training_set1["label"].astype(int).tolist(),
    tokenizer,
    label_dtype=torch.long,
 )
trainer_task1 = Trainer(
    model=model_task1,
    args=training_args_task1,
    train_dataset=train_dataset_task1,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_binary,
 )
trainer_task1.train()

test_dataset_task1 = TextDataset(
    tedf1["text"].tolist(),
    [0] * len(tedf1),
    tokenizer,
    label_dtype=torch.long,
 )
preds_output_task1 = trainer_task1.predict(test_dataset_task1)
preds_task1 = np.argmax(preds_output_task1.predictions, axis=1)

/vol/bitbucket/ks2222/NLP_IndividualCW/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/vol/bitbucket/ks2222/NLP_IndividualCW/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']).

Step,Training Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 15.57 GiB of which 13.38 MiB is free. Process 806719 has 790.00 MiB memory in use. Process 3186326 has 6.60 GiB memory in use. Process 3269778 has 4.46 GiB memory in use. Including non-PyTorch memory, this process has 3.57 GiB memory in use. Of the allocated memory 3.16 GiB is allocated by PyTorch, and 127.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

: 

In [ ]:
Counter(preds_task1)

In [ ]:
labels2file([[k] for k in preds_task1], 'task1.txt')

# Rebuild training set (Task 2)

In [ ]:
rows2 = [] # will contain par_id, label and text
for idx in range(len(trids)):
  parid = trids.par_id[idx]
  label = trids.label[idx]
  # select row from original dataset to retrieve the `text` value
  text = dpm.train_task1_df.loc[dpm.train_task1_df.par_id == parid].text.values[0]
  rows2.append({
      'par_id':parid,
      'text':text,
      'label':label
  })


In [ ]:
trdf2 = pd.DataFrame(rows2)

In [ ]:
trdf2

In [ ]:
trdf2.label = trdf2.label.apply(literal_eval)

# Rebuild test set (Task 2)

In [ ]:
rows2 = [] # will contain par_id, label and text
for idx in range(len(teids)):
  parid = teids.par_id[idx]
  label = teids.label[idx]
  #print(parid)
  # select row from original dataset to access the `text` value
  text = dpm.train_task1_df.loc[dpm.train_task1_df.par_id == parid].text.values[0]
  rows2.append({
      'par_id':parid,
      'text':text,
      'label':label
  })


In [ ]:
tedf2 = pd.DataFrame(rows2)

In [ ]:
tedf2

In [ ]:
tedf2.label = tedf2.label.apply(literal_eval)

# RoBERTa baseline for Task 2

In [ ]:
all_negs = trdf2[trdf2.label.apply(lambda x:sum(x) == 0)]
all_pos = trdf2[trdf2.label.apply(lambda x:sum(x) > 0)]

training_set2 = pd.concat([all_pos,all_negs[:round(len(all_pos)*0.5)]])

In [ ]:
training_set2

In [ ]:
training_args_task2 = TrainingArguments(
    output_dir="./tmp_task2",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=50,
    save_strategy="no",
    evaluation_strategy="no",
    report_to="none",
 )
model_task2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=7,
    problem_type="multi_label_classification",
 )
train_dataset_task2 = TextDataset(
    training_set2["text"].tolist(),
    training_set2["label"].tolist(),
    tokenizer,
    label_dtype=torch.float,
 )
trainer_task2 = Trainer(
    model=model_task2,
    args=training_args_task2,
    train_dataset=train_dataset_task2,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_multilabel,
 )
trainer_task2.train()

test_dataset_task2 = TextDataset(
    tedf2["text"].tolist(),
    [[0] * 7 for _ in range(len(tedf2))],
    tokenizer,
    label_dtype=torch.float,
 )
preds_output_task2 = trainer_task2.predict(test_dataset_task2)
probs_task2 = 1.0 / (1.0 + np.exp(-preds_output_task2.predictions))
preds_task2 = (probs_task2 >= 0.5).astype(int)

In [ ]:
labels2file(preds_task2, 'task2.txt')

## Prepare submission

In [ ]:
!cat task1.txt | head -n 10

In [ ]:
!cat task2.txt | head -n 10

In [ ]:
!zip submission.zip task1.txt task2.txt